# 막시무스: ODsay 없는 30분 대중교통 도달권
서울시 공식 버스·지하철 자료로 동대문구 334개 출발지의 08시·14시·19시 도달권을 계산합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/막시무스'  # 실제 경로로 수정
%cd $PROJECT_DIR
!pip -q install -r requirements-local-transit.txt

In [ ]:
from pathlib import Path
import pandas as pd
required = [
 'data/raw/서울시버스노선별정류소정보_20260902.xlsx',
 'data/raw/tpss_route_section_speedh_2026.08.24-08.30.zip',
 'data/raw/seoul_metro_timetable_20260616.csv',
 'data/raw/seoul_station_master_20260902.csv',
 'data/raw/tpss_sta_route_hturn_2026.08.24-08.30.zip',
 'data/raw/kscc_dx_trnsf_path_sum_20260903.zip',
 'data/external/odsay_origins.csv',
 'data/external/seoul_administrative_dongs_20260701.geojson']
for name in required:
    print(('OK  ' if Path(name).exists() else '없음'), name)

## 1. 실제 운행자료로 대기·환승시간 보정

In [ ]:
!python scripts/build_seoul_transit_network.py --bus-routes 'data/raw/서울시버스노선별정류소정보_20260902.xlsx' --bus-speeds data/raw/tpss_route_section_speedh_2026.08.24-08.30.zip --metro-timetable data/raw/seoul_metro_timetable_20260616.csv --station-master data/raw/seoul_station_master_20260902.csv --hour 8
!python scripts/calibrate_transit_times.py --bus-operations data/raw/tpss_sta_route_hturn_2026.08.24-08.30.zip --metro-stop-times data/interim/local_transit/metro_stop_times.csv --transfers data/raw/kscc_dx_trnsf_path_sum_20260903.zip --hours 8 14 19

## 2. 세 시간대 교통망과 30분 도달권 계산

In [ ]:
import subprocess
transfer_seconds = {8: 240, 14: 349, 19: 356}
for hour in [8, 14, 19]:
    subprocess.run(['python','scripts/build_seoul_transit_network.py','--bus-routes','data/raw/서울시버스노선별정류소정보_20260902.xlsx','--bus-speeds','data/raw/tpss_route_section_speedh_2026.08.24-08.30.zip','--metro-timetable','data/raw/seoul_metro_timetable_20260616.csv','--station-master','data/raw/seoul_station_master_20260902.csv','--hour',str(hour)], check=True)
    subprocess.run(['python','scripts/static_multimodal_accessibility.py','--hour',str(hour),'--minutes','30','--route-waits',f'data/interim/local_transit/calibration/route_waits_{hour:02d}.csv','--transfer-penalty-seconds',str(transfer_seconds[hour]),'--output-dir',f'data/processed/local_transit_30min/h{hour:02d}'], check=True)

## 3. 시간대 공통 후보 비교

In [ ]:
!python scripts/compare_accessibility_periods.py
result = pd.read_csv('data/processed/local_transit_30min/time_period_candidates.csv')
print(result.time_stability.value_counts())
display(result[result.time_stability == 'all_three_periods'].head(30))